In [ ]:
import os
import json
import torch
import pandas as pd
from datasets import Dataset
import evaluate
from transformers import (
    AutoModelForCausalLM,
    AutoTokenizer,
    BitsAndBytesConfig,
    TrainingArguments,
    pipeline
)
from peft import LoraConfig, get_peft_model, TaskType
from trl.trainer.sft_trainer import SFTTrainer

In [ ]:
def load_and_prepare_dataset(transcripts_path, summaries_path):
    def read_jsonl_safely(file_path):
        records = []
        with open(file_path, 'r', encoding='utf-8') as f:
            for line in f:
                cleaned_line = line.strip()
                if cleaned_line:  # Drop blank lines or whitespaces
                    records.append(json.loads(cleaned_line))
        return pd.DataFrame(records)

    df_transcripts = read_jsonl_safely(transcripts_path)
    df_summaries = read_jsonl_safely(summaries_path)
    
    if 'video_id' in df_summaries.columns:
        df_summaries = df_summaries.drop(columns=['video_id'])
        
    merged_df = pd.merge(df_transcripts, df_summaries, on='id')
    
    # Guarantee exact target counts (75 total)
    merged_df = merged_df.head(75).reset_index(drop=True)
    
    train_df = merged_df.iloc[:55].reset_index(drop=True)
    test_df = merged_df.iloc[55:75].reset_index(drop=True)
    
    print(f"Dataset split successfully! Train records: {len(train_df)}, Test records: {len(test_df)}")
    
    def apply_prompt_template(row):
        prompt = (
            "<start_of_turn>user\n"
            "تعليمات: قم بتلخيص النص المرئي التالي الخاص بمقطع الفيديو على شكل نقاط رئيسية مستخلصة دقيقة وموجزة 5 اسطر فقط(Bullet Points).\n"
            f"النص: {row['normalized_transcript']}<end_of_turn>\n"
            "<start_of_turn>model\n"
            f"{row['summary']}<end_of_turn>"
        )
        return prompt

    train_df['text'] = train_df.apply(apply_prompt_template, axis=1)
    test_df['text'] = test_df.apply(apply_prompt_template, axis=1)
    
    hf_train = Dataset.from_pandas(train_df[['id', 'text']])
    hf_test = Dataset.from_pandas(test_df[['id', 'normalized_transcript', 'summary']])
    
    return hf_train, hf_test

train_dataset, test_dataset = load_and_prepare_dataset("data/normalized_output.jsonl", "data/summarized_last.jsonl")

In [ ]:
import torch
import torch.nn as nn
from datasets import load_dataset
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
from transformers.models.gemma4 import modeling_gemma4
from peft import LoraConfig
from trl import SFTConfig, SFTTrainer


# Patching gemma 4 to fix issue preventing training
class PatchedClippableLinear(nn.Linear):
    def __init__(self, config, in_features, out_features):
        nn.Linear.__init__(self, in_features, out_features, bias=False)
        self.use_clipped_linears = getattr(config, "use_clipped_linears", False)
        
        if self.use_clipped_linears:
            self.register_buffer("input_min", torch.tensor(-float("inf")))
            self.register_buffer("input_max", torch.tensor(float("inf")))
            self.register_buffer("output_min", torch.tensor(-float("inf")))
            self.register_buffer("output_max", torch.tensor(float("inf")))

    def forward(self, x):
        if self.use_clipped_linears:
            x = torch.clamp(x, self.input_min, self.input_max)
        out = nn.Linear.forward(self, x)
        if self.use_clipped_linears:
            out = torch.clamp(out, self.output_min, self.output_max)
        return out

modeling_gemma4.Gemma4ClippableLinear = PatchedClippableLinear

MODEL_ID = "google/gemma-4-E4B-it"

tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.bfloat16,
    bnb_4bit_use_double_quant=True
)

model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    quantization_config=bnb_config,
    device_map="auto"
)

peft_config = LoraConfig(
    r=16,
    lora_alpha=32,
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"],
    exclude_modules=["vision_tower", "multi_modal_projector", "audio_tower"],
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM",
)

training_args = SFTConfig(
    output_dir="./gemma-transcript-summarizer2",
    per_device_train_batch_size=1,        
    gradient_accumulation_steps=1,
    learning_rate=2e-4,
    logging_steps=5,
    num_train_epochs=8,
    optim="paged_adamw_8bit",
    bf16=True,                             
    fp16=False,
    save_strategy="epoch",                 
    eval_strategy="no",                    
    report_to="none",                      
    max_length=2048,                       
)

trainer = SFTTrainer(
    model=model,
    train_dataset=train_dataset,
    peft_config=peft_config,
    args=training_args,
    processing_class=tokenizer,
)

trainer.train()

trainer.save_model("./final_summary_adapters2")
print("Model saved successfully.")

In [ ]:
model.config.use_cache = True
model.eval()

predictions = []
references = []

print("\nEvaluating Data.")

for idx, sample in enumerate(test_dataset):
    raw_transcript = sample['normalized_transcript']
    gold_summary = sample['summary']
    
    eval_prompt = (
        "<start_of_turn>user\n"
        "تعليمات: قم بتلخيص النص المرئي التالي الخاص بمقطع الفيديو تلخيصاً شاملاً ودقيقاً خمس اسطر فقط.\n"
        f"النص: {raw_transcript}<end_of_turn>\n"
        "<start_of_turn>model\n"
    )
    
    inputs = tokenizer(eval_prompt, return_tensors="pt").to(model.device)
    
    with torch.no_grad():
        output_tokens = model.generate(
        **inputs,
        max_new_tokens=300,
        do_sample=False,
        repetition_penalty=1.2,    
        no_repeat_ngram_size=3,      
        pad_token_id=tokenizer.eos_token_id
        )
    
    input_len = inputs['input_ids'].shape[1]
    generated_summary = tokenizer.decode(output_tokens[0][input_len:], skip_special_tokens=True).strip()
    
    predictions.append(generated_summary)
    references.append(gold_summary)

In [ ]:
from bert_score import scorer as bs_scorer

print("Evaluating BertScore")

arabic_bertscore_model = "asafaya/bert-base-arabic" 

bert_scorer = bs_scorer.BERTScorer(
    model_type=arabic_bertscore_model, 
    num_layers=9,
    lang="ar", 
    device=str(model.device)
)

bert_scorer._tokenizer.model_max_length = 512

P, R, F1 = bert_scorer.score(predictions, references)

mean_precision = P.mean().item()
mean_recall = R.mean().item()
mean_f1 = F1.mean().item()

print("\nBERTScore Summary:")
print(f"Precision: {mean_precision:.4f}")
print(f"Recall: {mean_recall:.4f}")
print(f"F1-Score:  {mean_f1:.4f}")